# 07 — Rerank Audit

End-to-end pipeline inspection: retrieval → rerank → MMR for a single playlist.
Side-by-side LightGBM vs CatBoost. Known-good audit and failure analysis.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from scipy.stats import kendalltau, spearmanr

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from etl.db import get_connection, init_schema
from paths import ARCHIVED_DIR, MODELS_DIR
from recommend.modules.classifiers import (
    build_rerank_features,
    load_classifier,
    rerank_candidates,
)
from recommend.modules.clustering import (
    build_cluster_features,
    filter_corpus_by_cluster,
    predict_cluster_probs,
)
from recommend.modules.similarity import (
    SIMILARITY_FEATURES,
    camelot_distance,
    compute_weighted_cosine,
    find_similar,
    playlist_centroid,
)
from recommend.preprocessing import build_feature_matrix
from utils.logging import configure_logging

configure_logging()
sns.set_theme(style="whitegrid", palette="muted")

conn = get_connection(read_only=False)
init_schema(conn)
corpus = build_feature_matrix(conn)
conn.close()

# Load model artifacts
gmm = joblib.load(MODELS_DIR / "gmm_corpus.pkl")
scaler = joblib.load(MODELS_DIR / "scaler_corpus.pkl")

print(f"Corpus: {corpus.shape}")

## 0. Select a Playlist

In [ ]:
enoa = pl.read_csv(ARCHIVED_DIR / "enoa.csv", null_values=["", "NA", "NaN"])

# List available playlists with their sizes
playlist_sizes = (
    enoa.group_by("playlist_name")
    .len()
    .sort("len", descending=True)
)
print("Available playlists:")
print(playlist_sizes)

# Pick the largest playlist for the audit
TARGET_PLAYLIST = playlist_sizes["playlist_name"][0]
print(f"\nSelected: {TARGET_PLAYLIST}")

In [ ]:
import re

def _playlist_slug(name: str) -> str:
    slug = name.lower().replace(" ", "_")
    return re.sub(r"[^a-z0-9_]", "", slug)

# Get playlist track IDs
playlist_ids = set(
    enoa.filter(pl.col("playlist_name") == TARGET_PLAYLIST)["id"].cast(pl.Utf8).to_list()
)
corpus_ids = set(corpus["id"].cast(pl.Utf8).to_list())
playlist_ids_in_corpus = playlist_ids & corpus_ids

print(f"Playlist tracks in corpus: {len(playlist_ids_in_corpus)} / {len(playlist_ids)}")

# Build playlist profile
id_list = corpus["id"].cast(pl.Utf8).to_list()
pos_mask = pl.Series([tid in playlist_ids_in_corpus for tid in id_list])
positives_df = corpus.filter(pos_mask)

centroid = playlist_centroid(positives_df, SIMILARITY_FEATURES)
key_modes = positives_df["key_mode"].to_list() if "key_mode" in positives_df.columns else []
modal_key = max(set(key_modes), key=key_modes.count) if key_modes else ""
mean_tempo = float(positives_df["tempo"].mean()) if "tempo" in positives_df.columns else 0.0

playlist_profile = {
    "centroid": centroid,
    "modal_key": modal_key,
    "mean_tempo": mean_tempo,
}
print(f"Modal key: {modal_key}, Mean tempo: {mean_tempo:.1f}")

## 1. Stage 1 — Cluster Filter

In [ ]:
# Compute query cluster probs from the playlist centroid
# Build a fake single-row DataFrame from centroid for cluster feature computation
centroid_dict = {feat: [float(centroid[i])] for i, feat in enumerate(SIMILARITY_FEATURES)}
# Add required columns for build_cluster_features with defaults
centroid_dict["key_mode"] = [modal_key]
centroid_dict["decade"] = ["2020s"]
centroid_dict["duration_ms_normalized"] = [0.5]
centroid_row = pl.DataFrame(centroid_dict)

query_features, _ = build_cluster_features(centroid_row, scaler, fit_scaler=False)
query_probs = predict_cluster_probs(query_features, gmm)

filtered = filter_corpus_by_cluster(corpus, query_probs[0], gmm, scaler, min_prob=0.05)

print(f"Full corpus:     {len(corpus):,}")
print(f"After cluster:   {len(filtered):,} ({len(filtered)/len(corpus)*100:.1f}%)")

if "cluster_prob" in filtered.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(filtered["cluster_prob"].to_numpy(), bins=50, edgecolor="white", alpha=0.8)
    ax.set_xlabel("cluster_prob")
    ax.set_title("Cluster Probability Distribution (filtered candidates)")
    plt.tight_layout()
    plt.show()

## 2. Stage 2 — Cosine Retrieval (Top-100)

In [ ]:
# Compute similarity scores for filtered corpus
X_filtered = filtered.select(SIMILARITY_FEATURES).to_numpy().astype(np.float64)
weights = np.ones(len(SIMILARITY_FEATURES))
sim_scores = compute_weighted_cosine(X_filtered, centroid, weights)

filtered = filtered.with_columns(pl.Series("similarity_score", sim_scores))

# Top 100 by similarity
top_100 = filtered.sort("similarity_score", descending=True).head(100)

print(f"Top-100 candidates retrieved")
print(f"Similarity range: [{top_100['similarity_score'].min():.4f}, {top_100['similarity_score'].max():.4f}]")

# How many actual playlist tracks in top-100?
top_100_ids = set(top_100["id"].cast(pl.Utf8).to_list())
hits = len(top_100_ids & playlist_ids_in_corpus)
print(f"Playlist tracks in top-100: {hits} / {len(playlist_ids_in_corpus)} (recall={hits/len(playlist_ids_in_corpus)*100:.1f}%)")

# Genre breakdown
if "gen_4" in top_100.columns:
    print(f"\nGenre breakdown (top-100):")
    print(top_100.group_by("gen_4").len().sort("len", descending=True))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
is_positive = [tid in playlist_ids_in_corpus for tid in top_100["id"].cast(pl.Utf8).to_list()]
colors = ["green" if p else "gray" for p in is_positive]
ax.bar(range(100), top_100["similarity_score"].to_numpy(), color=colors, alpha=0.7)
ax.set_xlabel("Rank")
ax.set_ylabel("Similarity score")
ax.set_title(f"Top-100 by Cosine Similarity (green = actual playlist member)")
plt.tight_layout()
plt.show()

## 3. Stage 3 — Rerank (LightGBM)

In [ ]:
slug = _playlist_slug(TARGET_PLAYLIST)
lgbm_clf = load_classifier(slug, MODELS_DIR)

if lgbm_clf is not None:
    reranked_lgbm = rerank_candidates(
        top_100, lgbm_clf, playlist_profile, model_type="lightgbm"
    )

    print(f"LightGBM rerank scores: [{reranked_lgbm['rerank_score'].min():.4f}, {reranked_lgbm['rerank_score'].max():.4f}]")

    # Show top 20
    display_cols = ["track_name", "rerank_score", "similarity_score", "gen_4", "decade"]
    display_cols = [c for c in display_cols if c in reranked_lgbm.columns]
    print("\nTop 20 after LightGBM rerank:")
    display(reranked_lgbm.select(display_cols).head(20))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(reranked_lgbm["rerank_score"].to_numpy(), bins=30, edgecolor="white", alpha=0.8)
    ax.set_xlabel("Rerank score")
    ax.set_title("LightGBM Rerank Score Distribution")
    plt.tight_layout()
    plt.show()
else:
    print(f"No classifier found for '{slug}' — run train.py first")
    reranked_lgbm = None

## 4. Stage 3b — Rerank (CatBoost)

In [ ]:
# CatBoost classifier — check if a catboost-trained artifact exists
# (they use the same slug, so you'd need to have trained with --model-type catboost)
# For comparison, we can retrain inline if needed

try:
    from recommend.modules.classifiers import train_playlist_classifier

    # Train a quick CatBoost classifier for comparison
    cat_pipeline, cat_metrics = train_playlist_classifier(
        corpus=corpus,
        playlist_track_ids=playlist_ids_in_corpus,
        scaler=scaler,
        gmm=gmm,
        gmm_scaler=scaler,
        model_type="catboost",
    )

    reranked_cat = rerank_candidates(
        top_100, cat_pipeline, playlist_profile, model_type="catboost"
    )

    print(f"CatBoost rerank scores: [{reranked_cat['rerank_score'].min():.4f}, {reranked_cat['rerank_score'].max():.4f}]")
    print(f"CatBoost metrics: {cat_metrics}")

    display_cols = ["track_name", "rerank_score", "similarity_score", "gen_4", "decade"]
    display_cols = [c for c in display_cols if c in reranked_cat.columns]
    print("\nTop 20 after CatBoost rerank:")
    display(reranked_cat.select(display_cols).head(20))
except Exception as exc:
    print(f"CatBoost comparison failed: {exc}")
    reranked_cat = None

## 5. Side-by-Side Top-20

In [ ]:
if reranked_lgbm is not None and reranked_cat is not None:
    lgbm_top20 = reranked_lgbm.head(20).select(["track_name", "rerank_score"]).rename({"rerank_score": "lgbm_score"})
    cat_top20 = reranked_cat.head(20).select(["track_name", "rerank_score"]).rename({"rerank_score": "cat_score"})

    print("Side-by-side Top-20:")
    print(f"{'Rank':>4}  {'LightGBM':40s} {'Score':>7}  {'CatBoost':40s} {'Score':>7}")
    print("-" * 105)
    for i in range(20):
        lgbm_name = lgbm_top20["track_name"][i] if i < len(lgbm_top20) else ""
        lgbm_score = lgbm_top20["lgbm_score"][i] if i < len(lgbm_top20) else 0
        cat_name = cat_top20["track_name"][i] if i < len(cat_top20) else ""
        cat_score = cat_top20["cat_score"][i] if i < len(cat_top20) else 0
        print(f"{i+1:>4}  {str(lgbm_name)[:40]:40s} {lgbm_score:>7.4f}  {str(cat_name)[:40]:40s} {cat_score:>7.4f}")

## 6. Rank Correlation

In [ ]:
if reranked_lgbm is not None and reranked_cat is not None:
    # Join on track ID to get aligned rankings
    lgbm_ranks = reranked_lgbm.with_row_index("lgbm_rank").select(["id", "lgbm_rank", "similarity_score"])
    cat_ranks = reranked_cat.with_row_index("cat_rank").select(["id", "cat_rank"])

    # Add cosine rank
    cosine_ranks = top_100.sort("similarity_score", descending=True).with_row_index("cosine_rank").select(["id", "cosine_rank"])

    merged = lgbm_ranks.join(cat_ranks, on="id").join(cosine_ranks, on="id")

    # Compute rank correlations
    lgbm_r = merged["lgbm_rank"].to_numpy()
    cat_r = merged["cat_rank"].to_numpy()
    cosine_r = merged["cosine_rank"].to_numpy()

    spearman_lc, _ = spearmanr(lgbm_r, cat_r)
    spearman_lo, _ = spearmanr(lgbm_r, cosine_r)
    spearman_co, _ = spearmanr(cat_r, cosine_r)

    kendall_lc, _ = kendalltau(lgbm_r, cat_r)

    print(f"Spearman rank correlations:")
    print(f"  LightGBM vs CatBoost:  {spearman_lc:.4f}")
    print(f"  LightGBM vs Cosine:    {spearman_lo:.4f}")
    print(f"  CatBoost vs Cosine:    {spearman_co:.4f}")
    print(f"\nKendall tau (LightGBM vs CatBoost): {kendall_lc:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].scatter(lgbm_r, cat_r, s=10, alpha=0.5)
    axes[0].set_xlabel("LightGBM rank")
    axes[0].set_ylabel("CatBoost rank")
    axes[0].set_title(f"LGBM vs Cat (ρ={spearman_lc:.3f})")
    axes[0].plot([0, 100], [0, 100], "r--", alpha=0.3)

    axes[1].scatter(cosine_r, lgbm_r, s=10, alpha=0.5)
    axes[1].set_xlabel("Cosine rank")
    axes[1].set_ylabel("LightGBM rank")
    axes[1].set_title(f"Cosine vs LGBM (ρ={spearman_lo:.3f})")
    axes[1].plot([0, 100], [0, 100], "r--", alpha=0.3)

    axes[2].scatter(cosine_r, cat_r, s=10, alpha=0.5)
    axes[2].set_xlabel("Cosine rank")
    axes[2].set_ylabel("CatBoost rank")
    axes[2].set_title(f"Cosine vs Cat (ρ={spearman_co:.3f})")
    axes[2].plot([0, 100], [0, 100], "r--", alpha=0.3)

    plt.suptitle("Rank Correlation: Cosine vs LightGBM vs CatBoost")
    plt.tight_layout()
    plt.show()

## 7. Known-Good Audit

Where do actual playlist tracks rank at each stage?

In [ ]:
# Track the known positives through the pipeline
full_sim_scores = compute_weighted_cosine(
    corpus.select(SIMILARITY_FEATURES).to_numpy().astype(np.float64),
    centroid, weights,
)
corpus_with_sim = corpus.with_columns(pl.Series("sim_score", full_sim_scores))
corpus_ranked = corpus_with_sim.sort("sim_score", descending=True).with_row_index("cosine_rank")

# Get ranks of playlist tracks
known_good = corpus_ranked.filter(
    pl.col("id").cast(pl.Utf8).is_in(list(playlist_ids_in_corpus))
)

if not known_good.is_empty():
    print(f"Known positives: {len(known_good)} tracks")
    print(f"\nCosine rank stats:")
    ranks = known_good["cosine_rank"].to_numpy()
    print(f"  Mean rank:   {ranks.mean():.0f}")
    print(f"  Median rank: {np.median(ranks):.0f}")
    print(f"  Best rank:   {ranks.min()}")
    print(f"  Worst rank:  {ranks.max()}")
    print(f"  In top-50:   {(ranks < 50).sum()}")
    print(f"  In top-100:  {(ranks < 100).sum()}")
    print(f"  In top-500:  {(ranks < 500).sum()}")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(ranks, bins=50, edgecolor="white", alpha=0.8)
    ax.axvline(np.median(ranks), color="red", linestyle="--", label=f"median={np.median(ranks):.0f}")
    ax.set_xlabel("Cosine rank (lower = more similar to playlist)")
    ax.set_title(f"Where Known Playlist Tracks Rank — {TARGET_PLAYLIST}")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 8. Failure Cases

Tracks ranked high by the reranker that are clearly wrong genre/era.

In [ ]:
if reranked_lgbm is not None:
    # Top-20 reranked tracks that are NOT in the playlist
    top20_lgbm = reranked_lgbm.head(20)
    false_positives = top20_lgbm.filter(
        ~pl.col("id").cast(pl.Utf8).is_in(list(playlist_ids_in_corpus))
    )

    if not false_positives.is_empty():
        inspect_cols = ["track_name", "rerank_score", "similarity_score", "gen_4", "decade", "first_genre"]
        inspect_cols = [c for c in inspect_cols if c in false_positives.columns]

        print(f"Non-playlist tracks in LightGBM top-20 ({len(false_positives)}):")
        display(false_positives.select(inspect_cols))

        # Compare their features to playlist centroid
        print("\nPlaylist centroid (SIMILARITY_FEATURES):")
        for i, feat in enumerate(SIMILARITY_FEATURES):
            fp_mean = false_positives[feat].mean() if feat in false_positives.columns else "N/A"
            print(f"  {feat:25s}  centroid={centroid[i]:.3f}  false_pos_mean={fp_mean}")
    else:
        print("All top-20 LightGBM tracks are actual playlist members!")

## 9. Rerank Impact Summary

In [ ]:
if reranked_lgbm is not None:
    # How much does reranking change the ordering vs cosine alone?
    cosine_order = top_100.sort("similarity_score", descending=True)["id"].cast(pl.Utf8).to_list()
    lgbm_order = reranked_lgbm["id"].cast(pl.Utf8).to_list()

    # Rank displacement for each track
    cosine_rank_map = {tid: i for i, tid in enumerate(cosine_order)}
    displacements = []
    for new_rank, tid in enumerate(lgbm_order):
        old_rank = cosine_rank_map.get(tid, -1)
        if old_rank >= 0:
            displacements.append(old_rank - new_rank)  # positive = promoted

    displacements = np.array(displacements)
    print(f"Rerank Impact Summary — {TARGET_PLAYLIST}")
    print(f"  Mean displacement: {displacements.mean():+.1f} positions")
    print(f"  Max promotion:     {displacements.max():+d} positions")
    print(f"  Max demotion:      {displacements.min():+d} positions")
    print(f"  Unchanged rank:    {(displacements == 0).sum()} tracks")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(displacements, bins=range(min(displacements)-1, max(displacements)+2), edgecolor="white", alpha=0.8)
    ax.axvline(0, color="red", linestyle="--", alpha=0.5)
    ax.set_xlabel("Rank displacement (positive = promoted by reranker)")
    ax.set_title("LightGBM Rerank: Position Change vs Cosine Baseline")
    plt.tight_layout()
    plt.show()